# Análisis de Centros — Docentes ↔ Estudiantes

Notebook dedicado a todo lo relacionado con `ID_CENTRO`:

1. Centros de docentes desfasados (2025 vs 2026) que **sí**/**no** tienen relación con estudiantes, y qué porcentaje del total de estudiantes representan.
2. Validación de si `ID_CENTRO` es la **misma entidad** en estudiantes y en docentes (comparando `tipo_centro`, `Rubro`, `dept_nombre` por código de centro común).
3. Centros de **estudiantes** desfasados (2025 vs 2026) que **sí**/**no** tienen relación con docentes, y qué porcentaje del total de docentes representan (dirección inversa de la sección 1).

Es autocontenido: carga sus propios datos y no depende de que otros notebooks hayan corrido antes (cada notebook es un kernel independiente).

In [ ]:
import pathsetup  # raíz del repo en sys.path (notebooks en reportes/)
from pathlib import Path

import pandas as pd

from compare_datasets_generic import (
    cargar,
    preparar_para_comparar,
    detectar_desfasajes,
    limpiar_duplicados_df,
    unir_anios,
    detectar_centros_relacionados,             # generica: sirve en ambas direcciones
    detectar_centros_relacionados_estudiantes, # alias fijo a estudiantes (compat)
    construir_tabla_centros,
    comparar_entidades_centro,
    limpiar_centros,             # limpieza -> df nuevo, elimina filas por codigo de centro
    pipeline_limpieza_centros,   # deteccion + limpieza encadenadas en un solo paso
)


In [ ]:
# ---- Config ----
# Carpeta fija: base DEFINITIVA entregada (2025-07-15).
CARPETA_DATOS = Path.home() / "Downloads" / "Datos Ceibal 2025-2026 ver final"
assert CARPETA_DATOS.exists(), f"No existe la carpeta de datos: {CARPETA_DATOS}"
SALIDA = CARPETA_DATOS / "processed"

print("Carpeta:", CARPETA_DATOS)

In [ ]:
# Carga de los 4 datasets limpios
doc25 = cargar(SALIDA / "docentes_2025_clean.csv")
doc26 = cargar(SALIDA / "docentes_2026_clean.csv")
est25 = cargar(SALIDA / "estudiantes_2025_clean.csv")
est26 = cargar(SALIDA / "estudiantes_2026_clean.csv")

print(f"docentes    2025: {len(doc25):,} filas | 2026: {len(doc26):,} filas")
print(f"estudiantes 2025: {len(est25):,} filas | 2026: {len(est26):,} filas")

## 1. Centros de docentes desfasados (2025 vs 2026)

Deduplicamos docentes primero (mismo criterio que `comparacion_inconsistencias.ipynb`) y detectamos qué códigos de `ID_CENTRO_docentes` aparecen en un solo año.


In [ ]:
doc25_sin_dups = limpiar_duplicados_df(doc25, verbose=True)
doc26_sin_dups = limpiar_duplicados_df(doc26, verbose=True)

doc25c, doc26c = preparar_para_comparar(doc25_sin_dups, doc26_sin_dups)

desfasajes_dedup = detectar_desfasajes(
    doc25c, doc26c, name1="2025", name2="2026", con_filas=True
)
desfasajes_df = desfasajes_dedup["categorias"]

centros_desfasados = desfasajes_df[desfasajes_df["columna"] == "ID_CENTRO_docentes"].copy()
print(f"\nCentros de docentes desfasados: {len(centros_desfasados)}")
centros_desfasados.head(20)

### 1a. ¿Esos centros están relacionados con estudiantes?

`detectar_centros_relacionados_estudiantes` chequea si cada centro desfasado también aparece en el dataset de estudiantes del mismo año, y agrega `n_filas_estudiantes`.

In [ ]:
centros_cruce = detectar_centros_relacionados_estudiantes(centros_desfasados, est25, est26)
centros_cruce.head(20)

In [ ]:
# Informe: centros RELACIONADOS con estudiantes (y % sobre el total de estudiantes de ese anio)
total_est = {"2025": len(est25), "2026": len(est26)}

relacionados = centros_cruce[centros_cruce["relacionado_con_estudiantes"]]
no_relacionados = centros_cruce[~centros_cruce["relacionado_con_estudiantes"]]

print("=== Centros de docentes desfasados RELACIONADOS con estudiantes ===\n")
for anio in ["2025", "2026"]:
    sub = relacionados[relacionados["solo_en"] == anio]
    n_est_cubiertos = int(sub["n_filas_estudiantes"].sum())
    pct = round(100 * n_est_cubiertos / total_est[anio], 2) if total_est[anio] else 0.0
    print(f"Año {anio}: {len(sub)} centros relacionados, cubren {n_est_cubiertos:,} filas de "
          f"estudiantes ({pct}% del total de estudiantes {anio} = {total_est[anio]:,})")

display(relacionados.sort_values("n_filas_estudiantes", ascending=False).reset_index(drop=True))

In [ ]:
# Informe: centros de docentes desfasados SIN relacion con estudiantes
print("=== Centros de docentes desfasados SIN relación con estudiantes ===\n")
for anio in ["2025", "2026"]:
    sub = no_relacionados[no_relacionados["solo_en"] == anio]
    print(f"Año {anio}: {len(sub)} centros sin relación con estudiantes "
          f"(de {len(centros_desfasados[centros_desfasados['solo_en'] == anio])} desfasados totales)")

display(no_relacionados.reset_index(drop=True))

## 2. ¿`ID_CENTRO` es la misma entidad en estudiantes y en docentes?

`estructura.py` renombra `ID_CENTRO` a `ID_CENTRO_docentes` / `ID_CENTRO_estudiantes` porque, según lo verificado en ese momento, la anonimización **no garantiza** que el mismo código represente el mismo centro físico entre ambos datasets.

Acá lo confirmamos con los datos: armamos un df de `[centro, tipo_centro, Rubro, dept_nombre]` por dataset (estudiantes 2025+2026 unidos vs. docentes 2025+2026 unidos) y comparamos, para los códigos de centro en común, si esos atributos coinciden. Si **no** coinciden, esa entidad de centro no es la misma → se informa (print) el conjunto completo.

In [ ]:
# Union 2025+2026 de cada tipo (DataFrames nuevos, no modifican doc25/doc26/est25/est26)
docentes_todos = unir_anios(doc25, doc26)
estudiantes_todos = unir_anios(est25, est26)

In [ ]:
# DataFrame por dataset: [centro, tipo_centro, Rubro, dept_nombre] (1 fila por centro)
centros_docentes = construir_tabla_centros(docentes_todos, "ID_CENTRO_docentes")
centros_estudiantes = construir_tabla_centros(estudiantes_todos, "ID_CENTRO_estudiantes")

print("Centros unicos docentes:", len(centros_docentes))
print("Centros unicos estudiantes:", len(centros_estudiantes))
centros_docentes.head()

In [ ]:
# Comparar por codigo de centro comun: tipo_centro / Rubro / dept_nombre deben coincidir
# si el codigo representa la MISMA entidad en estudiantes y en docentes.
centros_no_misma_entidad = comparar_entidades_centro(
    centros_estudiantes, centros_docentes, name_a="estudiantes", name_b="docentes"
)

print("\nCentros con el mismo codigo que NO son la misma entidad (estudiantes vs docentes):")
print(centros_no_misma_entidad)

## 3. Centros de estudiantes desfasados (2025 vs 2026)

Espejo de la Sección 1 pero para `ID_CENTRO_estudiantes`: deduplicamos estudiantes y detectamos qué códigos aparecen en un solo año.

In [ ]:
est25_sin_dups = limpiar_duplicados_df(est25, verbose=True)
est26_sin_dups = limpiar_duplicados_df(est26, verbose=True)

est25c, est26c = preparar_para_comparar(est25_sin_dups, est26_sin_dups)

desfasajes_dedup_est = detectar_desfasajes(
    est25c, est26c, name1="2025", name2="2026", con_filas=True
)
desfasajes_df_est = desfasajes_dedup_est["categorias"]

centros_desfasados_est = desfasajes_df_est[desfasajes_df_est["columna"] == "ID_CENTRO_estudiantes"].copy()
print(f"\nCentros de estudiantes desfasados: {len(centros_desfasados_est)}")
centros_desfasados_est.head(20)

### 3a. ¿Esos centros están relacionados con docentes?

Dirección inversa de la Sección 1a: `detectar_centros_relacionados` (genérica) chequea, para cada centro desfasado de estudiantes, si aparece en el dataset de **docentes** del mismo año.

In [ ]:
centros_cruce_est = detectar_centros_relacionados(
    centros_desfasados_est, doc25, doc26,
    col_centro_otro="ID_CENTRO_docentes", label="docentes",
)
centros_cruce_est.head(20)

In [ ]:
# Informe: centros de estudiantes desfasados RELACIONADOS con docentes (y % sobre el total de docentes de ese anio)
total_doc = {"2025": len(doc25), "2026": len(doc26)}

relacionados_est = centros_cruce_est[centros_cruce_est["relacionado_con_docentes"]]
no_relacionados_est = centros_cruce_est[~centros_cruce_est["relacionado_con_docentes"]]

print("=== Centros de estudiantes desfasados RELACIONADOS con docentes ===\n")
for anio in ["2025", "2026"]:
    sub = relacionados_est[relacionados_est["solo_en"] == anio]
    n_doc_cubiertos = int(sub["n_filas_docentes"].sum())
    pct = round(100 * n_doc_cubiertos / total_doc[anio], 2) if total_doc[anio] else 0.0
    print(f"Año {anio}: {len(sub)} centros relacionados, cubren {n_doc_cubiertos:,} filas de "
          f"docentes ({pct}% del total de docentes {anio} = {total_doc[anio]:,})")

display(relacionados_est.sort_values("n_filas_docentes", ascending=False).reset_index(drop=True))

In [ ]:
# Informe: centros de estudiantes desfasados SIN relacion con docentes
print("=== Centros de estudiantes desfasados SIN relación con docentes ===\n")
for anio in ["2025", "2026"]:
    sub = no_relacionados_est[no_relacionados_est["solo_en"] == anio]
    print(f"Año {anio}: {len(sub)} centros sin relación con docentes "
          f"(de {len(centros_desfasados_est[centros_desfasados_est['solo_en'] == anio])} desfasados totales)")

display(no_relacionados_est.reset_index(drop=True))

## 4. Limpieza de centros problemáticos → DataFrames nuevos

Completamos el patrón *reporte → detección → limpieza* también para centros:

| Detección | Limpieza → DataFrame nuevo |
|---|---|
| `detectar_centros_relacionados` (centros sin relación con el otro tipo) | `limpiar_centros` / `pipeline_limpieza_centros` |
| `comparar_entidades_centro` (centros que no son la misma entidad) | `limpiar_centros` / `pipeline_limpieza_centros` |

`pipeline_limpieza_centros` encadena ambas detecciones y devuelve, para un dataset y año dados, una **copia nueva** sin las filas de centros problemáticos (no modifica el original).

In [ ]:
# Docentes: pipeline deteccion + limpieza -> DataFrames nuevos por anio
doc25_limpio = pipeline_limpieza_centros(
    doc25, "ID_CENTRO_docentes", anio="2025",
    centros_no_relacionados=no_relacionados,
    centros_no_misma_entidad=centros_no_misma_entidad,
)
doc26_limpio = pipeline_limpieza_centros(
    doc26, "ID_CENTRO_docentes", anio="2026",
    centros_no_relacionados=no_relacionados,
    centros_no_misma_entidad=centros_no_misma_entidad,
)

print(f"\ndoc25: {len(doc25):,} -> {len(doc25_limpio):,} filas")
print(f"doc26: {len(doc26):,} -> {len(doc26_limpio):,} filas")

In [ ]:
# Estudiantes: mismo pipeline, para el centro sin relacion con docentes
est25_limpio = pipeline_limpieza_centros(
    est25, "ID_CENTRO_estudiantes", anio="2025",
    centros_no_relacionados=no_relacionados_est,
    centros_no_misma_entidad=centros_no_misma_entidad,
)
est26_limpio = pipeline_limpieza_centros(
    est26, "ID_CENTRO_estudiantes", anio="2026",
    centros_no_relacionados=no_relacionados_est,
    centros_no_misma_entidad=centros_no_misma_entidad,
)

print(f"\nest25: {len(est25):,} -> {len(est25_limpio):,} filas")
print(f"est26: {len(est26):,} -> {len(est26_limpio):,} filas")